# Practice: Unit Testing

Write your solutions in the **YOUR CODE** cells, then run **Validate** cells.

We use the built-in `unittest` and `unittest.mock` so everything runs inside the notebook (no separate `pytest` process needed). The setup cell adds a `run_suite(...)` helper that runs a `TestCase` and reports whether it passed.

In [ ]:
# Run this cell once per notebook — provides test validation helpers.
import asyncio
import threading


def check_equal(name, actual, expected):
    """Compare actual vs expected; print pass/fail."""
    if actual == expected:
        print(f"✓ {name}")
        return True
    print(f"✗ {name}")
    print(f"  Expected: {expected!r}")
    print(f"  Got:      {actual!r}")
    return False


def run_tests(cases):
    """Run a list of (name, callable) tuples. Callable should return the answer."""
    passed = 0
    for name, fn in cases:
        try:
            if check_equal(name, fn(), True):
                passed += 1
        except Exception as exc:
            print(f"✗ {name} — Error: {type(exc).__name__}: {exc}")
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("🎉 All tests passed!")
    return passed == total


def run_async(coro):
    """Run a coroutine safely, even inside a notebook's running event loop."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    result = {}

    def _runner():
        result["value"] = asyncio.run(coro)

    t = threading.Thread(target=_runner)
    t.start()
    t.join()
    return result["value"]


def run_suite(test_case_cls):
    """Run a unittest.TestCase subclass; return True if every test passed."""
    import unittest
    suite = unittest.TestLoader().loadTestsFromTestCase(test_case_cls)
    result = unittest.TextTestRunner(verbosity=0).run(suite)
    return result.wasSuccessful()


---

## Problem 1: code under test `Easy`

Implement `divide(a, b)` that returns `a / b`, but raises `ValueError` with the exact message `'cannot divide by zero'` when `b == 0`. The later problems test this function.

**Examples**

```
divide(10, 2) → 5.0;   divide(1, 0) → raises ValueError
```

In [ ]:
# YOUR CODE
def divide(a, b):
    pass


In [ ]:
check_equal('basic', divide(10, 2), 5.0)
try:
    divide(1, 0)
    print('✗ raises — no error was raised')
except ValueError as e:
    check_equal('raises', str(e), 'cannot divide by zero')


---

## Problem 2: assert that it raises `Medium`

Define `assert_raises(fn, exc)` that returns `True` if calling `fn()` raises an exception of type `exc`, and `False` otherwise (no exception, or a different type). This is the idea behind `assertRaises` / `pytest.raises`.

**Examples**

```
assert_raises(lambda: 1 / 0, ZeroDivisionError) → True
assert_raises(lambda: 1 + 1, ZeroDivisionError) → False
```

In [ ]:
# YOUR CODE
def assert_raises(fn, exc):
    pass


In [ ]:
check_equal('catches', assert_raises(lambda: 1 / 0, ZeroDivisionError), True)
check_equal('no_raise', assert_raises(lambda: 1 + 1, ZeroDivisionError), False)
check_equal('wrong_type', assert_raises(lambda: 1 / 0, ValueError), False)


---

## Problem 3: parametrized checks `Medium`

Define `run_cases(func, cases)` where `cases` is a list of `(args, expected)` tuples (`args` is itself a tuple). Call `func(*args)` for each case and return the **count** of cases where the result equals `expected`. This mirrors `pytest.mark.parametrize`.

**Examples**

```
run_cases(lambda x: x * 2, [((2,), 4), ((3,), 6)]) → 2
```

In [ ]:
# YOUR CODE
def run_cases(func, cases):
    pass


In [ ]:
check_equal('all_pass', run_cases(lambda x: x * 2, [((2,), 4), ((3,), 6)]), 2)
check_equal('some_fail', run_cases(lambda x: x * 2, [((2,), 4), ((3,), 99)]), 1)
check_equal('add', run_cases(lambda a, b: a + b, [((1, 2), 3), ((5, 5), 10)]), 2)


---

## Problem 4: a unittest TestCase `Medium`

Write a `unittest.TestCase` named `TestDivide` that tests the `divide` function from Problem 1:

- `test_normal`: use `self.assertEqual` to check `divide(10, 2) == 5.0`
- `test_zero`: use `self.assertRaises(ValueError)` to check `divide(1, 0)`

Then run it with `run_suite(TestDivide)`.

**Examples**

```
run_suite(TestDivide) → True
```

In [ ]:
# YOUR CODE
import unittest

class TestDivide(unittest.TestCase):
    def test_normal(self):
        pass

    def test_zero(self):
        pass


In [ ]:
check_equal('suite_passes', run_suite(TestDivide), True)
test_methods = [m for m in dir(TestDivide) if m.startswith('test_')]
check_equal('has_two_tests', len(test_methods), 2)


---

## Problem 5: mocking a dependency `Medium`

Implement `get_username(client, user_id)` that calls `client.fetch(user_id)` and returns the `'name'` field of the dict it returns. Writing it to take `client` as an argument makes it easy to test with a `Mock`.

**Examples**

```
client.fetch returns {'name': 'Asha'} → get_username(client, 7) == 'Asha'
```

In [ ]:
# YOUR CODE
def get_username(client, user_id):
    pass


In [ ]:
from unittest.mock import Mock
client = Mock()
client.fetch.return_value = {'name': 'Asha', 'id': 7}
check_equal('returns_name', get_username(client, 7), 'Asha')
check_equal('called_with', client.fetch.call_args.args, (7,))


---

## Problem 6: patching with mock `Medium`

Implement `is_weekend()` so it calls the helper `now()` (already defined) and returns `True` only on Saturday or Sunday. Hint: `weekday()` gives Monday=0 ... Sunday=6, so the weekend is `5` or `6`. Calling `now()` (instead of `datetime.now()` directly) lets the test **patch** the clock.

**Examples**

```
patched to a Saturday → True;   patched to a Wednesday → False
```

In [ ]:
# YOUR CODE
from datetime import datetime

def now():
    return datetime.now()

def is_weekend():
    pass


In [ ]:
from unittest.mock import patch
from datetime import datetime

with patch('__main__.now', return_value=datetime(2024, 1, 6)):  # Saturday
    check_equal('saturday', is_weekend(), True)
with patch('__main__.now', return_value=datetime(2024, 1, 3)):  # Wednesday
    check_equal('wednesday', is_weekend(), False)
